# Installing needed packages

In [ ]:
!pip install odfpy

# **Step 1:**
**Converting CSV to ODS**

In [ ]:
from odf.opendocument import load
from odf.table import Table, TableCell, TableRow
from odf.text import P
from odf import teletype
import pandas as pd

df = pd.read_csv('/content/churn-bigml-20.csv')
df.to_excel('customer_churn.ods', engine='odf', index=False)

file_name = 'customer_churn.ods'
doc = load(file_name)

# **Step 2:**
**Loading the sheet and displaying headers**

In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet
sheets = doc.getElementsByType(Table)
sheet = sheets[0]
table = sheet.getElementsByType(Table)
rows = sheet.getElementsByType(TableRow)

header_row = rows[0]
headers = [
    teletype.extractText(cell.getElementsByType(P)[0]) if cell.getElementsByType(P) else ''
    for cell in header_row.getElementsByType(TableCell)
]
print(f'Header Names: {headers}')

Header Names: ['State', 'Account length', 'Area code', 'International plan', 'Voice mail plan', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total day charge', 'Total eve minutes', 'Total eve calls', 'Total eve charge', 'Total night minutes', 'Total night calls', 'Total night charge', 'Total intl minutes', 'Total intl calls', 'Total intl charge', 'Customer service calls', 'Churn']


# **Step 4:**
**Displaying Sample Rows**

In [ ]:
for i, row in enumerate(rows[1:6], 1):

    cells = row.getElementsByType(TableCell)
    value = [
        teletype.extractText(cell.getElementsByType(P)[0]) if cell.getElementsByType(P) else ''
        for cell in cells
    ]
    print(f'{i}: {value}')

1: ['LA', '117', '408', 'No', 'No', '', '184.5', '97', '31.37', '351.6', '80', '29.89', '215.8', '90', '9.71', '8.7', '4', '2.35', '1', 'FALSE']
2: ['IN', '65', '415', 'No', 'No', '', '129.1', '137', '21.95', '228.5', '83', '19.42', '208.8', '111', '9.4', '12.7', '6', '3.43', '4', 'TRUE']
3: ['NY', '161', '415', 'No', 'No', '', '332.9', '67', '56.59', '317.8', '97', '27.01', '160.6', '128', '7.23', '5.4', '9', '1.46', '4', 'TRUE']
4: ['SC', '111', '415', 'No', 'No', '', '110.4', '103', '18.77', '137.3', '102', '11.67', '189.6', '105', '8.53', '7.7', '6', '2.08', '2', 'FALSE']
5: ['HI', '49', '510', 'No', 'No', '', '119.3', '117', '20.28', '215.1', '109', '18.28', '178.7', '90', '8.04', '11.1', '1', '3.0', '1', 'FALSE']


# **Step 5:**
**Data Cleaning**

In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P

# Cleaning the header row
clean_headers = []
for h in headers:
  clean_value = h.strip().lower().replace(' ', '_') if h is not None else ''
  clean_headers.append(clean_value)
print(clean_headers)

['state', 'account_length', 'area_code', 'international_plan', 'voice_mail_plan', 'number_vmail_messages', 'total_day_minutes', 'total_day_calls', 'total_day_charge', 'total_eve_minutes', 'total_eve_calls', 'total_eve_charge', 'total_night_minutes', 'total_night_calls', 'total_night_charge', 'total_intl_minutes', 'total_intl_calls', 'total_intl_charge', 'customer_service_calls', 'churn']


In [ ]:
# Cleaning the data rows
# Cleaning the data rows
clean_data = []
for row in rows[1:]:
    cells = row.getElementsByType(TableCell)
    value = [
        teletype.extractText(cell.getElementsByType(P)[0]) if cell.getElementsByType(P) else ''
        for cell in cells
    ]

    clean_row_values = []
    for val in value:
        clean_values = val.strip().lower() if val is not None else ''
        clean_row_values.append(clean_values)
    clean_data.append(clean_row_values)

# **Step 6:**
**Adding the clean data to ods file**

In [ ]:
clean_file = 'cleaned_customer_churn.ods'
clean_doc = OpenDocumentSpreadsheet()
clean_table = Table()

header_row = TableRow()
for h in clean_headers:
  cell = TableCell()
  cell.addElement(P(text=h))
  header_row.addElement(cell)
clean_table.addElement(header_row)

for rec in clean_data:
  row = TableRow()
  for val in rec:
    cell = TableCell()
    cell.addElement(P(text=str(val)))
    row.addElement(cell)
  clean_table.addElement(row)

clean_doc.spreadsheet.addElement(clean_table)
clean_doc.save('clean_file')

print(f'Cleaned file is saved as {clean_file}')

Cleaned file is saved as cleaned_customer_churn.ods


# **Questions:**

**Q1: Who cancels more: International or Voicemail users?**

In [ ]:
international_plan_yes = 0
international_churn_yes = 0

voice_mail_plan_yes = 0
voice_mail_churn_yes = 0

for row in clean_data:
  international_plan = row[3]
  voice_mail_plan = row[4]
  churn_status = row[19]

  if international_plan == 'yes':
    international_plan_yes += 1
    if churn_status == 'true':
      international_churn_yes += 1


  if voice_mail_plan == 'yes':
    voice_mail_plan_yes += 1
    if churn_status == 'true':
      voice_mail_churn_yes += 1

international_churn_rate = (international_churn_yes/international_plan_yes) * 100 if international_plan_yes > 0 else 0
voice_mail_churn_rate = (voice_mail_churn_yes/voice_mail_plan_yes) * 100 if voice_mail_plan_yes > 0 else 0

print(f'International churn rate is {international_churn_rate:.2f}%')
print(f'Voice mail churn rate is {voice_mail_churn_rate:.2f}%')


International churn rate is 35.85%
Voice mail churn rate is 7.94%


**Q2: Which rigion has the highest churn rate?**

In [ ]:
from collections import defaultdict
state_churn = defaultdict(list)
for rec in clean_data:
  state = rec[0]
  churn = rec[19]
  if churn == 'true':
    state_churn[state].append(churn)
state_counts_list = []
highest_count = 0
winning_state = ''
for state, churn_length in state_churn.items():
  churn_count = len(churn_length)
  state_counts_list.append([state, churn_count])
  if highest_count < churn_count:
    highest_count = churn_count
    winning_state = state
top_10_highest_churn_states = sorted(state_counts_list, key=lambda item: item[1], reverse=True)[:10]
print(f'Top 10 states with highes churn rate are {top_10_highest_churn_states}')
print(winning_state)

Top 10 states with highes churn rate are [['id', 4], ['or', 4], ['ca', 4], ['mt', 4], ['nj', 4], ['wa', 4], ['in', 3], ['ny', 3], ['ks', 3], ['ri', 3]]
id


**Q3: Which region contacts tech support more?**

In [ ]:
state_contactsupport = defaultdict(list)

for rec in clean_data:
  state = rec[0]
  if rec[18] != '':
    contact_support = int(rec[18])
    if contact_support > 0:
      state_contactsupport[state].append(contact_support)
  state_call_totals = []
for state, call_list in state_contactsupport.items():
  total_calls = sum(call_list)
  state_call_totals.append([state, total_calls])
top_10_states_to_contact_support = sorted(state_call_totals, key=lambda item: item[1], reverse=True)[:10]
print(top_10_states_to_contact_support)

[['az', 40], ['nm', 35], ['tx', 34], ['in', 33], ['id', 30], ['wa', 30], ['ms', 29], ['ct', 29], ['nj', 29], ['wi', 28]]


# **Task:**
**Finding the Heavy Users. **

In [ ]:
# make a list
# make sum of day, eve, night minutes and stor them in list
# find the average of total min for each state
# find the highest average total min by state

state_user_min = defaultdict(list)

for rec in clean_data:
  if rec[7] != '' and rec[10] != '' and rec[13] != '':
    day_min = float(rec[6])
    eve_min = float(rec[9])
    night_min = float(rec[12])
    state = rec[0]

    total_minutes = sum([day_min, eve_min, night_min])
    state_user_min[state].append(total_minutes)

  state_user_min_avg = []
  for state, total_min in state_user_min.items():
    avg = (sum(total_min) / len(total_min))
    state_user_min_avg.append([state, avg])
top_10_users_avg_minutes = sorted(state_user_min_avg, key=lambda item: item[1], reverse=True)[:10]
print(top_10_users_avg_minutes)

[['mn', 622.6285714285714], ['fl', 619.0111111111112], ['or', 615.075], ['nc', 609.1], ['nd', 605.0444444444444], ['ct', 603.8866666666667], ['in', 603.8705882352941], ['ca', 603.3399999999999], ['hi', 602.7777777777778], ['ar', 601.9124999999999]]


# **Veifier:**

In [ ]:
florida_min = 0.0
florida_customer= 0

for rec in clean_data:
  if rec[6] != '' and rec[9] != '' and rec[12] != '':
    day_min = float(rec[6])
    eve_min = float(rec[9])
    night_min = float(rec[12])

    if rec[0] == 'fl':
      florida_customer += 1
      total_min = sum([day_min, eve_min, night_min])
      florida_min += total_min

avg = florida_min / florida_customer
print(avg)

619.0111111111112
